# Lab 02-04 — Projecting and clustering the 768-dim embedding space

**Track 02 · Embeddings** — making the embedding space visible. Embeddings from `BAAI/bge-base-en-v1.5` live in a 768-dimensional space that no human can look at directly. This lab makes that space visible with the two classic tools:

* **PCA** (`sklearn.decomposition.PCA`) — a *linear* projection that keeps the directions of largest variance; its explained-variance ratio tells us how much structure survives in just 2 dimensions.
* **UMAP** (`umap.UMAP`) — a *non-linear* manifold projection that preserves local neighbourhood structure, which is usually what retrieval actually cares about.

Then we cluster with `KMeans` (k=5) in all three spaces — raw 768-dim embeddings, the UMAP projection, and the PCA projection — and compare the `silhouette_score` of each. The silhouette score measures how separated the clusters are: raw embeddings are the baseline, and the projections tell us how much of that separation is visible in 2D.

Teaching point: a 768-dim space is *sparse and high-volume* — distances in it are much less intuitive than in 2D/3D. Projections are lossy (PCA keeps only the variance it reports), but they reveal the *topical* structure: Wikipedia passages about the same subject end up near each other, which is exactly the signal retrieval exploits.

This notebook is **self-contained**: it imports LangChain, numpy, pandas, scikit-learn, umap-learn, and matplotlib directly — no repo component library. The BGE embedder is built right here as a small inline wrapper over `HuggingFaceEmbeddings` with `normalize_embeddings=True`, which is exactly how the shared `src/embeddings/bge.py` component works underneath. The scatter plot is saved to `Data/scratch-02-embeddings/umap_scatter.png` (never under `src/`).


## Setup

One prerequisite must hold before this notebook will run:

- **rag-mini-wikipedia on disk** — `Data/corpus/rag-mini-wikipedia/passages.parquet`, already fetched by the repo's manifest-verified fetchers.

No repo imports are needed: everything this notebook uses comes from `langchain-huggingface`, `numpy`, `pandas`, `scikit-learn`, `umap-learn`, and `matplotlib`. The imports cell walks up to the repo root and cd's into it, because a notebook has no `__file__` — so every `Data/...` path resolves exactly like the lab script. Unlike the Curriculum notebook, there is no `sys.path` trick: nothing is imported from `src/`.

The next cell installs the notebook-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# below is a no-op if you have run `pip install -r requirements.txt`).
#   sentence-transformers -> local BGE embeddings
#   langchain-huggingface -> HuggingFaceEmbeddings (the universal embedder class)
#   pandas                -> reads the passages.parquet corpus
#   numpy                 -> vector math (n x 768 arrays, norms)
#   scikit-learn          -> PCA, KMeans, silhouette_score
#   umap-learn            -> UMAP manifold projection
#   matplotlib            -> the 2D scatter (Agg backend, saved to PNG)
%pip install -q sentence-transformers langchain-huggingface pandas numpy scikit-learn umap-learn matplotlib


In [ ]:
# Bootstrap: stdlib imports + repo-root walk (no sys.path tricks).
from __future__ import annotations

import os
import time
from pathlib import Path

# Agg backend: render the PNG headlessly (no display server needed).
import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
from sklearn.cluster import KMeans  # noqa: E402
from sklearn.decomposition import PCA  # noqa: E402
from sklearn.metrics import silhouette_score  # noqa: E402

# LangChain + numpy/pandas/sklearn — the only libraries this notebook needs.
# Nothing is imported from the repo's src/ component library.
from langchain_core.embeddings import Embeddings  # noqa: E402
from langchain_huggingface import HuggingFaceEmbeddings  # noqa: E402

try:
    import umap  # noqa: E402

    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False


class BGEEmbedding(Embeddings):
    """Inline BGE wrapper — mirrors src/embeddings/bge.py.

    bge models require normalized embeddings for cosine similarity; the
    universal HuggingFaceEmbeddings class provides that via encode_kwargs.
    """

    def __init__(self, model_name: str = "BAAI/bge-base-en-v1.5"):
        self.model = HuggingFaceEmbeddings(
            model_name=model_name,
            encode_kwargs={"normalize_embeddings": True},
        )

    def embed_query(self, text: str) -> list[float]:
        return self.model.embed_query(text)

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return self.model.embed_documents(texts)


# A notebook has no __file__, so walk up from the cwd to the repo root and
# cd into it — Data/... paths then resolve exactly like the lab script.
REPO_ROOT = Path.cwd()
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "curriculum").is_dir() and (_candidate / "NoteBooks").is_dir():
        REPO_ROOT = _candidate
        break
os.chdir(REPO_ROOT)


## 1. Configuration

Everything that keeps this lab fast but still meaningful is a named constant. `CORPUS_PATH` points at the rag-mini-wikipedia passages parquet; `N_PASSAGES = 400` is the deterministic `head(400)` subset (keeps runtime under ~3 minutes); `N_CLUSTERS = 5` is the KMeans k; and `OUT_PLOT` is where the UMAP scatter is saved — `Data/scratch-02-embeddings/umap_scatter.png` (the scratch series is additive-only, so the plot never lands under `src/`).


In [ ]:
# --------------------------------------------------------------------------
# 1. Configuration — tweak these to rerun
# --------------------------------------------------------------------------
CORPUS_PATH = Path("Data/corpus/rag-mini-wikipedia/passages.parquet")
N_PASSAGES = 400  # deterministic subset: head(400), keeps runtime under ~3 min
N_CLUSTERS = 5
OUT_PLOT = Path("Data/scratch-02-embeddings/umap_scatter.png")


## 2. Load + embed

`load_passages` returns the first `n` passage texts; `embed_passages` embeds them all once with BGE into an `n x 768` float array; `norm_stats` returns (min, mean, max) L2 norm per row (BGE normalizes, so these sit at ~1.0); `coord_ranges` returns the x/y extents of a 2D coordinate array.


In [ ]:
# --------------------------------------------------------------------------
# 2. Load + embed
# --------------------------------------------------------------------------
def load_passages(path: Path, n: int) -> list[str]:
    """Return the first ``n`` passage texts from the rag-mini corpus."""
    df = pd.read_parquet(path)
    return list(df["passage"].head(n))


def embed_passages(texts: list[str]) -> np.ndarray:
    """Embed all passages once with BGE; returns an ``n x 768`` float array."""
    vectors = BGEEmbedding().embed_documents(texts)
    return np.asarray(vectors, dtype=np.float32)


def norm_stats(matrix: np.ndarray) -> tuple[float, float, float]:
    """Return (min, mean, max) L2 norm per row."""
    norms = np.linalg.norm(matrix, axis=1)
    return float(norms.min()), float(norms.mean()), float(norms.max())


def coord_ranges(matrix: np.ndarray) -> tuple[tuple[float, float], tuple[float, float]]:
    """Return ((x_min, x_max), (y_min, y_max)) of the 2D coordinates."""
    return (
        (float(matrix[:, 0].min()), float(matrix[:, 0].max())),
        (float(matrix[:, 1].min()), float(matrix[:, 1].max())),
    )


## 3. Projections

`project_pca` projects to 2D with PCA (random_state=42) and returns the coordinates plus the explained-variance ratio summed over the two components. `project_umap` projects to 2D with UMAP (random_state=42 for a reproducible layout).


In [ ]:
# --------------------------------------------------------------------------
# 3. Projections
# --------------------------------------------------------------------------
def project_pca(matrix: np.ndarray) -> tuple[np.ndarray, float]:
    """Project to 2D with PCA; returns (coords, explained-variance sum)."""
    pca = PCA(n_components=2, random_state=42)
    coords = pca.fit_transform(matrix)
    return coords, float(pca.explained_variance_ratio_.sum())


def project_umap(matrix: np.ndarray) -> np.ndarray:
    """Project to 2D with UMAP (random_state=42 for reproducible layout)."""
    reducer = umap.UMAP(n_components=2, random_state=42)
    return np.asarray(reducer.fit_transform(matrix))


## 4. Clustering

`cluster_kmeans` runs KMeans (k=5, n_init=10, random_state=42) on a matrix and returns `(labels, cluster sizes in label order, silhouette score)` — run once per space (raw, PCA, UMAP).


In [ ]:
# --------------------------------------------------------------------------
# 4. Clustering
# --------------------------------------------------------------------------
def cluster_kmeans(matrix: np.ndarray) -> tuple[np.ndarray, list[int], float]:
    """KMeans (k=5, n_init=10) on ``matrix``.

    Returns (labels, cluster sizes in label order, silhouette score).
    """
    labels = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10).fit_predict(
        matrix
    )
    sizes = [int(np.sum(labels == i)) for i in range(N_CLUSTERS)]
    return labels, sizes, float(silhouette_score(matrix, labels))


## 5. Plot

`save_umap_scatter` draws the UMAP projection coloured by the UMAP-space cluster labels and saves the figure to `OUT_PLOT` (Agg backend, so it renders headlessly).


In [ ]:
# --------------------------------------------------------------------------
# 5. Plot
# --------------------------------------------------------------------------
def save_umap_scatter(coords: np.ndarray, labels: np.ndarray, path: Path) -> None:
    """Scatter of the UMAP projection coloured by the UMAP-space cluster labels."""
    fig, ax = plt.subplots(figsize=(8, 6))
    scatter = ax.scatter(
        coords[:, 0],
        coords[:, 1],
        c=labels,
        cmap="tab10",
        s=12,
        alpha=0.8,
    )
    ax.set_title("UMAP projection of BGE embeddings (768-dim -> 2D), KMeans k=5")
    ax.set_xlabel("UMAP dim 1")
    ax.set_ylabel("UMAP dim 2")
    fig.colorbar(scatter, ax=ax, label="cluster")
    fig.tight_layout()
    fig.savefig(path, dpi=110)
    plt.close(fig)


## 6. Run the experiment — embed, project, cluster, plot

`run_experiment` embeds the 400 passages with BGE (timed), projects with PCA and UMAP (timed, UMAP when available), clusters each space, saves the UMAP scatter, and returns everything the demo and the gate need — no printing happens here.


In [ ]:
# --------------------------------------------------------------------------
# 6. Run the experiment — embed, project, cluster, plot
# --------------------------------------------------------------------------
def run_experiment() -> dict:
    t0 = time.perf_counter()
    texts = load_passages(CORPUS_PATH, N_PASSAGES)
    embeddings = embed_passages(texts)
    t_embed = time.perf_counter() - t0

    pca_coords, var_ratio = project_pca(embeddings)
    umap_coords = None
    t_umap = 0.0
    if UMAP_AVAILABLE:
        t0 = time.perf_counter()
        umap_coords = project_umap(embeddings)
        t_umap = time.perf_counter() - t0

    baseline_labels, baseline_sizes, baseline_sil = cluster_kmeans(embeddings)
    pca_labels, pca_sizes, pca_sil = cluster_kmeans(pca_coords)
    umap_labels, umap_sizes, umap_sil = None, None, None
    if UMAP_AVAILABLE:
        umap_labels, umap_sizes, umap_sil = cluster_kmeans(umap_coords)

    plot_kb = 0.0
    if UMAP_AVAILABLE:
        OUT_PLOT.parent.mkdir(parents=True, exist_ok=True)
        save_umap_scatter(umap_coords, umap_labels, OUT_PLOT)
        plot_kb = OUT_PLOT.stat().st_size / 1024

    return {
        "passages": texts,
        "embeddings": embeddings,
        "norms": norm_stats(embeddings),
        "t_embed": t_embed,
        "t_umap": t_umap,
        "pca_coords": pca_coords,
        "var_ratio": var_ratio,
        "umap_coords": umap_coords,
        "clusters": {
            "raw 768-dim": (baseline_sizes, baseline_sil),
            "PCA 2D": (pca_sizes, pca_sil),
            "UMAP 2D": (umap_sizes, umap_sil),
        },
        "umap_available": UMAP_AVAILABLE,
        "plot_kb": plot_kb,
    }


## 7. Demo — print the artifact

`print_demo(exp)` prints the artifact from five angles: the setup (corpus, passages, k, UMAP availability); the embedding step (shape, row norms, timing); the projections (PCA explained variance and coordinate ranges; UMAP fit time and coordinate ranges); the KMeans clustering per space (cluster sizes + silhouette); the saved plot; and a takeaway on how little variance fits in two PCA axes yet how crisp the same structure looks in 2D.


In [ ]:
# --------------------------------------------------------------------------
# 7. Demo — print the artifact
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 04 — projecting and clustering the 768-dim embedding space")
    print("=" * 66)

    print("\n[1] Setup")
    print(f"    corpus          : {CORPUS_PATH}")
    print(f"    passages        : {N_PASSAGES}")
    print(f"    clusters (k)    : {N_CLUSTERS}")
    print(f"    umap available  : {exp['umap_available']}")

    print("\n[2] Embed (BAAI/bge-base-en-v1.5, local)")
    embeddings = exp["embeddings"]
    n_min, n_mean, n_max = exp["norms"]
    print(f"    embedded {embeddings.shape[0]} passages in {exp['t_embed']:.1f}s")
    print(f"    shape     : {embeddings.shape[0]} x {embeddings.shape[1]} (768-dim)")
    print(f"    row norms : min={n_min:.4f} mean={n_mean:.4f} max={n_max:.4f} "
          f"(BGE normalizes -> ~1.0)")

    print("\n[3] Projections (768-dim -> 2D)")
    pca_coords = exp["pca_coords"]
    pca_x, pca_y = coord_ranges(pca_coords)
    print(f"    PCA  : explained variance (2 components) = {exp['var_ratio']:.3f}")
    print(f"           coords x in [{pca_x[0]:+.2f}, {pca_x[1]:+.2f}], "
          f"y in [{pca_y[0]:+.2f}, {pca_y[1]:+.2f}]")

    if exp["umap_available"]:
        umap_coords = exp["umap_coords"]
        umap_x, umap_y = coord_ranges(umap_coords)
        print(f"    UMAP : fit in {exp['t_umap']:.1f}s")
        print(f"           coords x in [{umap_x[0]:+.2f}, {umap_x[1]:+.2f}], "
              f"y in [{umap_y[0]:+.2f}, {umap_y[1]:+.2f}]")
    else:
        print("    UMAP : SKIP - umap-learn not installed; "
              "running clustering on raw + PCA only")

    print(f"\n[4] KMeans k={N_CLUSTERS} per space (sizes + silhouette)")
    for space, (sizes, sil) in exp["clusters"].items():
        if sizes is None:
            continue
        print(f"    {space:<10} : sizes={sizes}  silhouette={sil:.3f}")

    print("\n[5] Plot")
    if exp["umap_available"]:
        print(f"    saved UMAP scatter -> {OUT_PLOT} ({exp['plot_kb']:.0f} KB)")
    else:
        print("    SKIP - no UMAP projection, nothing to plot")

    print("\n[6] Takeaway")
    if exp["umap_available"]:
        baseline_sizes, baseline_sil = exp["clusters"]["raw 768-dim"]
        _pca_sizes, pca_sil = exp["clusters"]["PCA 2D"]
        _umap_sizes, umap_sil = exp["clusters"]["UMAP 2D"]
        print(f"    Only {exp['var_ratio']:.0%} of the variance fits in two PCA axes, yet")
        print("    UMAP still recovers compact topical neighbourhoods. The raw")
        print(f"    768-dim silhouette is low ({baseline_sil:.2f}) - distances in high-dim")
        print("    space are diluted by the curse of dimensionality - while the")
        print(f"    2D projections make the same structure crisp ({pca_sil:.2f} PCA, "
              f"{umap_sil:.2f}")
        print("    UMAP). That visible geometry is exactly what vector")
        print("    similarity retrieval exploits.")


## 8. Verification gate

`verify_gate(exp)` enforces the lab's hard checks: the embedding matrix is `N_PASSAGES x 768` with unit row norms (BGE's contract); the PCA explained-variance ratio lands in `(0, 1)`; every clustered space yields `N_CLUSTERS` clusters whose sizes sum to `N_PASSAGES` with silhouettes in `[-1, 1]`; and when UMAP is available, its coordinates are `N_PASSAGES x 2` and the scatter was actually written to `OUT_PLOT`. Every check should print PASS.


In [ ]:
# --------------------------------------------------------------------------
# 8. Verification gate
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []

    embeddings = exp["embeddings"]
    checks.append((f"embeddings shape {embeddings.shape[0]} x {embeddings.shape[1]} "
                   f"({N_PASSAGES} x 768)",
                   embeddings.shape == (N_PASSAGES, 768)))
    n_min, n_mean, n_max = exp["norms"]
    checks.append((f"row norms ~1.0 (min={n_min:.4f} mean={n_mean:.4f} "
                   f"max={n_max:.4f})",
                   abs(n_min - 1.0) < 1e-2 and abs(n_mean - 1.0) < 1e-2
                   and abs(n_max - 1.0) < 1e-2))
    checks.append((f"PCA explained variance in (0, 1): {exp['var_ratio']:.3f}",
                   0.0 < exp["var_ratio"] < 1.0))

    for space, (sizes, sil) in exp["clusters"].items():
        if sizes is None:
            continue
        checks.append((f"{space}: {N_CLUSTERS} clusters with {N_PASSAGES} total points",
                       len(sizes) == N_CLUSTERS and sum(sizes) == N_PASSAGES))
        checks.append((f"{space}: silhouette in [-1, 1] ({sil:.3f})",
                       -1.0 <= sil <= 1.0))

    if exp["umap_available"]:
        checks.append((f"UMAP coords are {N_PASSAGES} x 2",
                       exp["umap_coords"].shape == (N_PASSAGES, 2)))
        checks.append((f"plot saved to {OUT_PLOT} ({exp['plot_kb']:.0f} KB)",
                       exp["plot_kb"] > 0))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

A couple of minutes of local embedding + UMAP fit (BGE cached on disk) — no downloads, no API calls. `exp` holds everything the demo and gate need.


In [ ]:
exp = run_experiment()


### Demo — the artifact

The projections and clusters: PCA explained variance, UMAP fit, per-space KMeans sizes + silhouettes, and the saved UMAP scatter (view `Data/scratch-02-embeddings/umap_scatter.png`).


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces. If any line shows FAIL, check the parquet file is intact.


In [ ]:
verify_gate(exp)
